# Redes Neurais em Grafos (GNNs)

**Autor:** Lucas Candinho

# Importando Bibliotecas

Aqui apenas importamos as bibliotecas, nada demais.

In [1]:
import numpy as np
np.random.seed(33)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
torch.manual_seed(33)

# O que são grafos?

Grafos são estruturas matemáticas compostas por um conjunto de objetos e suas conexões uns com os outros. Os objetos são representados pelos nós/vértices (do inglês: node/vertices) do grafo, enquanto suas relações são representadas por linhas que conectam os nós, chamadas arestas (do inglês: edges) [1,2].

Note que as relações podem ser tanto direcionais (relações não necessáriamente recíprocas) ou não-direcionais (todas as relações são, por definição, reciprocas) [1,2].

<img src="grafos.svg" width="500">

`(A) Um grafo direcional; (B) Um grafo não-direcional`

`Fonte: Produção do Autor.`

Grafos podem ter, também, atributos imbuidos em suas relações, nas arestas.

<img src="grafo_atr.svg" width="500">

`Um grafo não direcional com atributos de relação`

`Fonte: Produção do Autor.`

## Representação por Matriz de Adjacência

Embora existam diferentes maneiras de se representar um grafo em python, aqui está o método da matriz de adjacência.

Por exemplo, os grafos A e B podem ser representados por:

In [2]:
A = [
    [0, 1, 0, 0, 1],
    [0, 0, 0, 0, 1],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0]
]

B = [
    [0, 1, 0, 0, 1],
    [1, 0, 0, 0, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 1, 0, 0],
    [1, 1, 1, 0, 0]
]

Note que, por ser não-direcional $B[i][j] = B[j][i] \; , \; i,j \in \mathbb{Z}$.

Veja, também que o objeto em $i,j$ podem ser qualquer estrutura no python, podendo representar relações complexas para além da simples vizinhança de dois objetos.

Assim, podemos representar o grafo C:

In [3]:
C = [
    [0, ["quadrado", "triângulo"], 0, 0, ["quadrado"]],
    [0, ["quadrado", "triângulo"], 0, 0, ["triangulo"]],
    [0, 0, 0, ["triângulo", "quadrado"], ["triangulo"]],
    [0, 0, ["triângulo", "quadrado"], 0, 0],
    [["quadrado"], ["triangulo"], ["triangulo"], 0, 0]
]

Note, entretanto, que essa maneira de representação pode gerar conflitos, dado que um mesmo grafo pode ter mais de uma representação por matriz de adjacência [4]. Por exemplo, no grafo B:

In [4]:
B_original = [
#    1  2  3  4  5
    [0, 1, 0, 0, 1],  # 1
    [1, 0, 0, 0, 1],  # 2
    [0, 0, 0, 0, 1],  # 3
    [0, 0, 0, 0, 1],  # 4
    [1, 1, 1, 1, 0],  # 5
]

B_reordenado = [
#    3  1  4  2  5
    [0, 0, 0, 0, 1],  # 3
    [0, 0, 0, 1, 1],  # 1
    [0, 0, 0, 0, 1],  # 4
    [0, 1, 0, 0, 1],  # 2
    [1, 1, 1, 1, 0],  # 5
]

Como nem sempre sabemos com que formato receberemos a matriz de adjacência, outras formas de representação podem ser melhores, dependendo do caso.

Para mais informações sobre como a matriz de adjacência funciona e pode ser usada, confira a referência [6].

## Representação por Lista de Adjacência

Há, entretanto, uma maneira de se representar grafos de uma maneira única. Na lista de adjacência cada nó mapeia diretamente para o conjunto de seus vizinhos, sem depender de uma ordenação global dos nós. Veja como o grafo B fica nessa representação:

In [5]:
B_la = {
    1: [2, 5],
    2: [1, 5],
    3: [5],
    4: [5],
    5: [1, 2, 3, 4]
}

Para mais informações sobre como a lista de adjacência funciona e pode ser usada, confira a referência [3].

## Materiais extras sobre grafos

Existem outras propriedades, representações e maneiras de visualizar essas estruturas, mas, por estar fora do escopo desse notebook, serão omitidas por brevidade. Entretanto, o assunto "grafos" é bem diverso, e tem suas particularidades e sutilizas tanto na matemática quanto na computação que podem atiçar a curiosidade de um leitor. Por isso, deixo aqui alguns materiais para maior exploração.

**Graph Neural Networks Designed for Different Graph Types: A Survey** (artigo) $-$ Josephine, et al (2023) $-$ disponível em https://arxiv.org/pdf/2204.03080. O artigo fala, primariamente (como indica o título) sobre redes neurais de grafos. Entretanto, sua introdução fundamenta bem o que é um grafo e quais formatos ele pode possuir (bem como propriedades, exemplos, etc) dado que cita redes para cada um dos tipos descritos.

**A Gentle Introduction to Graph Neural Networks** (artigo web) $-$ Sanchez-Lengeling, et al. (Distill) $-$ disponível em https://distill.pub/2021/gnn-intro/. Novamente, um artigo que trata sobre redes neurais como tópico principal, mas, novamente, um bom material para explorar os grafos, com muitos exemplos (inclusive, exemplos interativos).

Note que a maioria das referências deste trabalho possui, de alguma maneira, uma introdução a grafos imbuida. Caso sinta a vontade, explore todas e veja como cada um explica esse conceito.

# Que tipos de problema podemos resolver com grafos?

No contexto de trabalhar com grafos, existem 3 níveis de problema que podemos resolver [3]:
- Nível de Grafo
- Nível de Vértice/Nó
- Nível de Aresta

No nível de grafo, o _input_ são grafos e o _output_ é alguma informação sobre o grafo como um todo. Nesse tipo de problema queremos classificar o grafo em si; por exemplo dado o grafo de uma molécula, preveja se haverá ligação ou não [3]. Note que, caso haja um grafo onde cada nó seja uma molécula e o objetivo seja classificar a molécula, caso a molécula (nó) seja representada como um grafo (numa situação grafo dentro de grafo) o problema pode ser dividido em nível de Nó com um problema a nível de Grafo dentro dele [4].

No nível de nó, o foco é (intutivamente) os nós. O _input_ é um grafo com nós vazios (não rotulados), e o _output_ é o grafo com os nós preenchidos (rotulados) [3]. Esse tipo de problema pode ser tanto de classificação (para dados categóricos) quanto de regressão (para dados numéricos). Há também, um terceiro tipo de problema que é o de clusterização de nós [4].

Por fim, no nível de aresta, o objetivo é prever as relações entre os nós (no caso, as arestas). Esses problemas são, em sua maioria, de classificação [3,4]. Dentro dessa categoria há os problemas de previsão de aparecimento de novas relações/arestas, por exemplo no caso de prever uma conexão entre dois usuários de uma rede social, isso se chama de _link-prediction_ [5].

Podemos ver alguns exemplos de problemas abaixo:

![svg](problemas.svg)

`Exemplos de problema em (A) nível de Nó; (B) nível de Aresta; (C) nível de Grafo`

`Fonte: Disponível em [4]`

# Por que outras redes neurais não trabalham bem com grafos?

Segundo Prince [6] existem três dificuldades inerentes que surgem ao trabalhar com grafos e redes neurais:
- A topologia dos grafos é variável e é difícil construir redes que são suficientemente expressivas e conseguem lidar com essa particularidade
- Grafos podem ser enormes (imagine um grafo de relações dos usuários do X [antigo Twitter] com seus milhões de usuários)
- Pode haver apenas um único grafo monolítico disponível, de modo que o protocolo usual de treinar com muitos exemplos de dados e testar com novos dados nem sempre é apropriado.

## Equivariância e Invariância

Como vimos, a lista de adjacência representa o grafo de forma naturalmente invariante à ordem dos nós, cada nó mapeia diretamente para seus vizinhos, independente de qualquer indexação global. Essa intuição se formaliza em dois requisitos centrais que qualquer GNN deve satisfazer [7].

A indexação dos nós em um grafo é arbitrária, qualquer permutação dos índices não altera o grafo em si. GNNs devem respeitar essa propriedade [7].

Para tarefas de nível de nó e aresta, exigimos equivariância: se permutarmos os índices dos nós, as representações internas dos nós $-$ chamadas de _embeddings_, vetores numéricos que codificam as características de cada nó e de sua vizinhança $-$ devem ser permutadas da mesma forma. O grafo continua o mesmo, e cada nó continua carregando a mesma informação, só a ordem mudou.

Para tarefas de nível de grafo, exigimos invariância: a predição final sobre o grafo como um todo não deve mudar com nenhuma permutação dos nós. Isso é alcançado naturalmente pela camada de agregação global (_readout_), que combina as representações de todos os nós de forma independente da ordem.

## Grafos enormes, _Neighbor Sampling_ e _Graph Partitioning_

Em grafos muito grandes e densos, agregar informações de todos os vizinhos de cada nó a cada camada se torna computacionalmente inviável. Isso ocorre porque cada nó depende de seus vizinhos na camada anterior, que por sua vez dependem dos seus próprios vizinhos $-$ isso forma o que se chama de _k-hop neighborhood_, onde `k` é o número de camadas da rede. Em casos extremos, cada nó pode estar no campo receptivo de todos os outros, problema conhecido como _graph expansion_ [6].

<img src="k-hopping.png" width="500">

`Escolhe-se um subconjunto de nós rotulados na camada de saída e então encontra-se todos os nós no K-hop neighborhood (campo receptivo). Apenas esse subgrafo é necessário para treinar o batch. Em grafos densamente conectados, isso pode reter uma grande proporção do grafo. O brilho representa a distância do nó original [6].`

`Fonte: Adaptado de [6]`

Duas abordagens foram desenvolvidas para contornar esse problema [6]:

1. _Neighbor Sampling_: a cada iteração de treino, apenas um subconjunto aleatório de vizinhos é amostrado em cada camada, mantendo o custo computacional fixo e independente do grau dos nós.
2. _Graph Partitioning_: o grafo original é particionado em subgrafos menores e desconexos antes do treinamento, utilizando algoritmos que minimizam o número de arestas removidas. Cada subgrafo pode então ser tratado como um batch independente, ou subconjuntos deles podem ser combinados reinstaurando as arestas originais entre eles.

<img src="n_sampling.png" width="500">

`Neighbor Sampling. Partindo da camada final, selecionamos um subconjunto de vizinhos na camada anterior e um subconjunto dos vizinhos desses na camada seguinte, restringindo o tamanho do grafo para o batch. O brilho representa a distância do nó original [6].`

`Fonte: Adaptado de [6]`

<img src="g_partitioning.png" width="500">

`Graph partitioning. a) Grafo de entrada original. b) O grafo é particionado em subgrafos menores minimizando o número de arestas removidas (linhas pontilhadas). c-d) Os subgrafos resultantes podem ser usados diretamente como batches independentes para treino no cenário transdutivo, nesse caso, há quatro batches possíveis, um por subgrafo. e) Por outro lado, combinações de subgrafos podem ser usadas como batches, reinstaurando as arestas originais entre eles; com pares de subgrafos, haveria seis batches possíveis [6].`

`Fonte: Adaptado de [6]`

## Modelos Transdutivos vs Indutivos

A terceira dificuldade introduz dois tipos distintos de aprendizado: Transdutivo e Indutivo. 

No cenário indutivo o modelo aprende uma regra a partir de dados rotulados e a aplica a novos dados não vistos. No cenário transdutivo, não há separação clara: o modelo é treinado e testado sobre o mesmo grafo, vendo todos os nós durante o treino mas sem os rótulos dos nós de teste [6].

Um exemplo é que, em um grafo de citações científicas, alguns artigos têm rótulos de área (física, biologia, etc.) e outros não. O objetivo é rotular os nós desconhecidos; entretanto, os dados de treino e teste estão conectados no mesmo grafo [6].

<img src="Indutivo_vs_Transdutivo.png" width="500">

`Aprendizado indutivo vs. transdutivo em classificação de nós. a) Cenário indutivo: são fornecidos múltiplos grafos de treino com rótulos conhecidos. Após o treinamento, o modelo recebe grafos de teste inéditos e deve atribuir rótulos a cada nó. b) Cenário transdutivo: há um único grafo monolítico onde alguns nós possuem rótulos conhecidos e outros não. O modelo é treinado para prever corretamente os rótulos conhecidos e então as predições são examinadas nos nós desconhecidos. Nós coloridos são conhecidos, enquanto nós com o símbolo <?> são desconhecidos (para o modelo) [6].`

`Fonte: Traduzido de [6]`

# Quais tipos de redes neurais de grafos existem?

Existe uma multiplicidade de propriedades de grafos diferentes, cada um servindo para representar um problema ou situação diferente. Dada a estrutura de cada um, existem algoritmos diferente para lidar com cada um deles. O artigo Graph Neural Networks Designed for Different Graph Types: A Survey (2023) de Josephine, et al cita diversos (muitos mesmo) tipos de grafos e algoritmos para lidar com cada um deles (por sinal, no final do artigo eles citam a motivação para a escolha de cada um deles).

Dado a quantidade que existem, não iremos citar todos (novamente, nosso intuito não é escrever um livro).

Nesse notebook citaremos 3 tipos de GNNs diferentes (que utilizam do mesmo _framework_, o MPNN), uma para cada tipo de problema anteriormente citado. Antes disso, porém, citarei alguns pontos notáveis sobre GNNs.

## GNNs como uma generalização de CNNs

Redes neurais convolucionais (CNNs) funcionam bem para imagens porque imagens são grafos com estrutura regular:, de forma que cada _pixel_ tem exatamente os mesmos vizinhos em posições fixas, o que permite compartilhar pesos de forma sistemática via convolução. GNNs surgem da ideia de generalizar essa ideia para domínios irregulares, onde cada nó pode ter um número arbitrário de vizinhos sem ordenação natural [8].

Nesse sentido, uma GNN pode ser vista como uma convolução sobre um grafo $-$ em vez de deslizar um filtro sobre posições fixas de uma grade, o filtro opera sobre vizinhanças locais de cada nó, de forma invariante à ordem dos vizinhos. Além disso, _Transformers_ podem ser interpretados como um caso especial de GNN aplicada a um grafo completo, onde cada _token_ presta atenção a todos os outros [8,9].

## Abordagens Espectrais vs. Espaciais

Historicamente, as primeiras GNNs foram desenvolvidas no domínio espectral. A ideia central é definir convoluções sobre grafos a partir do espectro do Laplaciano do grafo $-$ a matriz que captura a estrutura de conectividade em termos de suas frequências. Para uma camada espectral, a regra de propagação é dada por [7]:

$$H^{(l+1)} = \sigma\left(\tilde{D}^{-\frac{1}{2}}\tilde{A}\tilde{D}^{-\frac{1}{2}}H^{(l)}W^{(l)}\right)$$

onde $\tilde{A} = A + I$ é a matriz de adjacência com auto-conexões, $\tilde{D}$ é a matriz de grau correspondente, $W^{(l)}$ é uma matriz de pesos aprendível e $\sigma$ é uma função de ativação. O termo $\tilde{D}^{-\frac{1}{2}}\tilde{A}\tilde{D}^{-\frac{1}{2}}$ é o Laplaciano normalizado do grafo, que atua como um filtro global sobre toda a estrutura do grafo [7].

Embora matematicamente elegante, essa abordagem tem limitações práticas, os filtros espectrais dependem da estrutura global do grafo e não generalizam naturalmente para grafos com topologias diferentes das vistas no treino [10].

As abordagens espaciais, por outro lado, operam diretamente sobre as vizinhanças locais dos nós, sem recorrer ao espectro global. Cada nó agrega informações de seus vizinhos imediatos de forma análoga à convolução local em imagens. Essa abordagem é mais flexível, computacionalmente mais eficiente e generaliza naturalmente para grafos não vistos [10]. 

## O _Framework_ MPNN e a Hierarquia de GNNs

Apesar da diversidade de arquiteturas, Gilmer et al. [13] mostraram que a grande maioria das GNNs modernas pode ser unificada sob o _framework Message Passing Neural Network_ (MPNN). Ele opera em três etapas que se repetem por $K$ camadas:

1. _Message_: cada nó $v$ recebe mensagens de seus vizinhos $u \in \mathcal{N}(v)$, computadas por uma função de mensagem $\psi$:

$$m_v^{(k)} = \bigoplus_{u \in \mathcal{N}(v)} \psi\left(h_v^{(k-1)}, h_u^{(k-1)}, e_{uv}\right)$$

onde $h_v^{(k)}$ é o _embedding_ do nó $v$ na camada $k$, $e_{uv}$ são os atributos da aresta entre $u$ e $v$, e $\bigoplus$ é uma operação de agregação, tipicamente soma, média ou máximo.

2. _Update_: cada nó atualiza seu _embedding_ combinando sua representação atual com as mensagens recebidas, via uma função de atualização $\phi$:

$$h_v^{(k)} = \phi\left(h_v^{(k-1)}, m_v^{(k)}\right)$$

3. _Readout_: para tarefas de nível de grafo, após $K$ camadas, uma função de agregação global $R$ combina os _embeddings_ de todos os nós em uma representação única do grafo:

$$\hat{y} = R\left(\left\{h_v^{(K)} \mid v \in G\right\}\right)$$

Dentro desse _framework_, Josephine et al. [10] estabelecem uma hierarquia formal entre os tipos de camadas GNN:

$$\text{message-passing} \supseteq \text{attention} \supseteq \text{convolution}$$

Nas camadas convolucionais, os vizinhos são agregados com pesos fixos e pré-definidos $c_{u,v}$, sem aprendizado sobre a importância relativa de cada vizinho [10]:

$$h_u^{(k)} = \phi\left(h_u, \bigoplus_{v \in \mathcal{N}(u)} c_{u,v}\, \psi(h_v)\right)$$

Nas camadas de atenção, esses pesos são substituídos por uma função aprendível $\text{att}(h_u, h_v)$, calculada dinamicamente para cada par de nós [10]:

$$h_u^{(k)} = \phi\left(h_u, \bigoplus_{v \in \mathcal{N}(u)} \text{att}(h_u, h_v)\, \psi(h_v)\right)$$

As camadas de _message passing_ são o caso mais geral: as mensagens $m_{uv}$ são computadas explicitamente a partir das representações dos dois nós, sem restrições sobre a forma dessa função [10]:

$$h_u^{(k)} = \phi\left(h_u, \bigoplus_{v \in \mathcal{N}(u)} \psi(h_u, h_v)\right)$$

Arquiteturas diferentes $-$ GCN, GAT, GraphSAGE, etc. $-$ são instâncias desse framework, distinguindo-se pela escolha de $\psi$, $\phi$, $\bigoplus$ e $R$.

## Expressividade e o Teste de Weisfeiler-Leman

Xu et al. [12] desenvolveram um framework teórico para responder a questão de qual o limite da expressividade de uma GNN, conectando o poder de discriminação das GNNs ao teste de Weisfeiler-Leman (WL), um algoritmo clássico para testar isomorfismo de grafos.

O teste WL funciona de forma análoga ao _message passing_, iterativamente, cada nó agrega os rótulos de seus vizinhos e atualiza seu próprio rótulo com base nessa agregação. Dois grafos são considerados não-isomórficos se, em alguma iteração, a distribuição de rótulos diferir entre eles [12].

Xu et al. [12] demonstram, pelo Lema 2, que qualquer GNN baseada em agregação de vizinhança é no máximo tão poderosa quanto o teste WL, ou seja, se uma GNN consegue distinguir dois grafos, o teste WL também consegue. Essa é uma limitação fundamental; GNNs padrão como GCN e GraphSAGE, por usarem agregadores de média e máximo respectivamente, são estritamente menos poderosas que o teste WL e falham em distinguir estruturas surpreendentemente simples, como ilustrado na figura abaixo.

<img src="min_max_WL.png" width="500">

`Exemplos de estruturas de grafo que agregadores de média e máximo falham em distinguir [12].`

`Fonte: Adaptado de [12]`

### Existe alguma GNN que atinge o limite do teste WL?

O Teorema 3 de Xu et al. [12] responde que sim $-$ desde que as funções de agregação e readout sejam injetivas, ou seja, mapeiem vizinhanças diferentes para representações diferentes. Isso motivou o desenvolvimento do GIN (Graph Isomorphism Network), que usa uma agregação por soma com MLP [12]:

$$h_v^{(k)} = \text{MLP}^{(k)}\left(\left(1 + \epsilon^{(k)}\right) \cdot h_v^{(k-1)} + \sum_{u \in \mathcal{N}(v)} h_u^{(k-1)}\right)$$

onde $\epsilon$ é um parâmetro aprendível ou fixo. A soma é injetiva sobre multiconjuntos $-$ ao contrário da média e do máximo $-$ garantindo que o GIN seja tão expressivo quanto o teste WL [12].

# DiffPool 

GNNs tradicionais são inerentemente _flat_, propagando informação ao longo das arestas do grafo, mas incapazes de aprender representações hierárquicas $-$ uma limitação especialmente problemática para classificação de grafos, onde o objetivo é prever um rótulo associado ao grafo inteiro [11].

O DiffPool, como proposto por Ying et al., propõe um módulo de _pooling_ diferenciável que aprende a agrupar nós em clusters de forma hierárquica e _end-to-end_, podendo ser combinado com qualquer arquitetura GNN [11]. A ideia central é análoga ao _pooling_ espacial em CNNs; assim como CNNs operam iterativamente sobre representações cada vez mais grosseiras de uma imagem, o DiffPool opera sobre representações cada vez mais _coarsened_ do grafo.

## Como funciona

O DiffPool empilha $L$ módulos GNN em forma hierárquica. A cada camada $l$, dois GNNs são aplicados em paralelo sobre a matriz de adjacência $A^{(l)}$ e as _features_ dos nós $X^{(l)}$ [11]:

1. GNN de _embedding_ | gera _embeddings_ dos nós:

$$Z^{(l)} = \text{GNN}_{l,\text{embed}}(A^{(l)}, X^{(l)})$$

2. GNN de _pooling_ | gera uma matriz de atribuição _soft_ $S^{(l)} \in \mathbb{R}^{n_l \times n_{l+1}}$, onde cada linha representa um nó e cada coluna representa um _cluster_ no próximo nível:

$$S^{(l)} = \text{softmax}\left(\text{GNN}_{l,\text{pool}}(A^{(l)}, X^{(l)})\right)$$

Com $S^{(l)}$ e $Z^{(l)}$ em mãos, o grafo é _coarsened_ para o próximo nível [11]:

$$X^{(l+1)} = S^{(l)^T} Z^{(l)} \in \mathbb{R}^{n_{l+1} \times d}$$

$$A^{(l+1)} = S^{(l)^T} A^{(l)} S^{(l)} \in \mathbb{R}^{n_{l+1} \times n_{l+1}}$$

A equação de $X^{(l+1)}$ agrega os _embeddings_ dos nós de acordo com as atribuições de _cluster_. A equação de $A^{(l+1)}$ gera uma nova matriz de adjacência pesada representando a força de conectividade entre cada par de _clusters_.

### Função de Perda

A perda total é composta por três termos [13]:

1. Perda de classificação | Entropia cruzada entre as predições e os rótulos verdadeiros.

2. Perda de _link prediction_ auxiliar | encoraja nós próximos a serem agrupados juntos:

$$\mathcal{L}_{LP} = \left\| A^{(l)} - S^{(l)} S^{(l)^T} \right\|_F$$

3. Regularização de entropia | encoraja atribuições próximas de _one-hot_, tornando os _clusters_ bem definidos:

$$\mathcal{L}_E = \frac{1}{n} \sum_{i=1}^{n} H(S_i)$$

onde $H$ é a entropia e $S_i$ é a $i$-ésima linha de $S$.

A perda total é: 

$$\mathcal{L} = \mathcal{L}_{\text{class}} + \mathcal{L}_{LP} + \mathcal{L}_E$$

### Hiperparâmetros

| Hiperparâmetro | Descrição |
|---|---|
| $L$ | Número de camadas de _pooling_ |
| $n_{l+1}$ | Número de _clusters_ em cada camada (tipicamente 25% de $n_l$) |
| $K$ | Número de camadas internas de _message passing_ por módulo GNN |
| $d$ | Dimensão dos _embeddings_ |

## Implementando com `pytorch`

### Criação de Dados

Os dados são gerados por duas funções que constroem grafos sintéticos. Eles podem ser de dois tipos, com comunidades ou homogêneos.

<img src="diffpool.png" width="500">

`(a) Um grafo com duas comunidades distintas, uma em azul e outra em vermelho, sendo conectados em uma ponte; (B) Um grafo homogêneo, sem comunidades.`

`Fonte: Produção do Autor.`

`criar_grafo_com` retorna sempre o mesmo grafo com comunidades, a variação entre os três exemplos de treino vem apenas da inicialização aleatória dos pesos do modelo, não dos dados. 

In [6]:
def criar_grafo_com():
    """
    Grafo com 2 comunidades densas, Classe 1
    """
    A = torch.tensor([
        [0,1,1,0,0,0],
        [1,0,1,0,0,0],
        [1,1,0,1,0,0],
        [0,0,1,0,1,1],
        [0,0,0,1,0,1],
        [0,0,0,1,1,0],
    ], dtype=torch.float)
    
    X = torch.tensor([
        [1.0,0.9,0.1,0.1],
        [0.9,1.0,0.1,0.2],
        [0.8,0.9,0.2,0.1],
        [0.1,0.2,0.9,1.0],
        [0.1,0.1,1.0,0.9],
        [0.2,0.1,0.9,0.8],
    ])
    
    return A, X, torch.tensor([1])

`criar_grafo_hom` recebe uma _seed_ para gerar features levemente diferentes a cada chamada, simulando três instâncias distintas do mesmo tipo de grafo. 

In [7]:
def criar_grafo_hom(seed):
    """
    Grafo completo homogêneo sem comunidades,  Classe 0.
    Todos os nós conectados a todos.
    """
    A = torch.ones(6, 6) - torch.eye(6)
    torch.manual_seed(seed)
    X = torch.ones(6, 4) * 0.5 + torch.randn(6, 4) * 0.05
    return A, X, torch.tensor([0])

O dataset final contém 6 grafos balanceados: 3 de cada classe.

In [8]:
dataset = [
    criar_grafo_com(),
    criar_grafo_com(),
    criar_grafo_com(),
    criar_grafo_hom(33),
    criar_grafo_hom(42),
    criar_grafo_hom(77),
]

### Definindo as camadas

O modelo é composto por três classes que se encaixam hierarquicamente.

A classe base é `GNNLayer`, uma camada de _message passing_ com agregação por média normalizada. Isso é equivalente a uma instância do framework MPNN onde $\psi$ é uma transformação linear e $\bigoplus$ é a média ponderada pelo grau. 

A normalização pelo grau evita que nós com muitos vizinhos dominem a agregação. O `clamp(min=1)` previne divisão por zero para nós isolados.

In [9]:
class GNNLayer(nn.Module):
    """
    Camada GNN — instância de MPNN com agregação por média normalizada.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, A, X):
        deg = A.sum(dim=-1, keepdim=True).clamp(min=1)
        agg = (A @ X) / deg
        return F.relu(self.W(agg))

`DiffPoolLayer` instancia dois `GNNLayer` separados, um para gerar os _embeddings_ Z e outro para gerar a matriz de atribuição S. 

O _softmax_ aplicado à saída do `gnn_pool` garante que cada linha de S seja uma distribuição de probabilidade válida sobre os _clusters_. 

As equações (3) e (4) de Ying et al. [11] são implementadas diretamente nas duas linhas de produto matricial.

In [10]:
class DiffPoolLayer(nn.Module):
    """
    Camada DiffPool.
    Dois GNNs separados: um para embedding, um para atribuição de clusters.
    """
    def __init__(self, in_dim, embed_dim, n_clusters):
        super().__init__()
        self.gnn_embed = GNNLayer(in_dim, embed_dim)
        self.gnn_pool = GNNLayer(in_dim, n_clusters)

    def forward(self, A, X):
        Z = self.gnn_embed(A, X)
        S = F.softmax(self.gnn_pool(A, X), dim=-1)

        X_next = S.T @ Z
        A_next = S.T @ A @ S

        return X_next, A_next, S

`DiffPool` combina todos os componentes em um classificador. 

O _forward_ executa três etapas em sequência: (1) a camada `DiffPool` _coarsens_ o grafo, um GNN final opera sobre o grafo _coarsened_, e o _readout_ por soma agrega os _embeddings_ dos _clusters_ em um único vetor que representa o grafo inteiro.

Esse vetor é então passado ao classificador MLP. O método _loss_ computa a perda total como soma das três contribuições discutidas na seção teórica.

In [11]:
class DiffPool(nn.Module):
    """
    Classificador de grafos com DiffPool.
    """
    def __init__(self, in_dim, embed_dim, n_clusters, n_classes):
        super().__init__()
        self.diffpool = DiffPoolLayer(in_dim, embed_dim, n_clusters)
        self.gnn_final = GNNLayer(embed_dim, embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes)
        )

    def forward(self, A, X):
        X_c, A_c, S = self.diffpool(A, X)
        Z_final = self.gnn_final(A_c, X_c)
        graph_emb = Z_final.sum(dim=0)
        out = self.classifier(graph_emb)
        return out, S

    def loss(self, out, y, A, S):
        # Entropia Cruzada
        L_class = F.cross_entropy(out.unsqueeze(0), y)

        # Link prediction
        L_lp = torch.norm(A - S @ S.T, p='fro')

        # Entropia
        S_clip = S.clamp(min=1e-10)
        L_ent  = -(S_clip * S_clip.log()).sum(dim=-1).mean()

        return L_class + L_lp + L_ent

### Treino do Modelo

O treino segue o protocolo padrão do `pytorch`, para cada grafo do _dataset_, computa o _forward pass_, calcula a perda total, propaga os gradientes e atualiza os pesos. 

Note que cada grafo é processado individualmente, não há _batching_, pois grafos diferentes têm tamanhos potencialmente distintos, como discutido na seção de dificuldades. 

O otimizador Adam com taxa de aprendizado 0.01 é uma escolha arbitrária para esse exemplo didático.

In [12]:
model = DiffPool(in_dim=4, embed_dim=8, n_clusters=2, n_classes=2)
optimizer = Adam(model.parameters(), lr=0.01)

for epoch in range(1, 11):
    model.train()
    total_loss = 0
    for A, X, y in dataset:
        optimizer.zero_grad()
        out, S = model(A, X)
        loss = model.loss(out, y, A, S)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 2 == 0:
        print(f"Epoch {epoch:3d} | Perda total: {total_loss:.4f}")

Epoch   2 | Perda total: 26.1993
Epoch   4 | Perda total: 25.9211
Epoch   6 | Perda total: 25.5820
Epoch   8 | Perda total: 24.8852
Epoch  10 | Perda total: 23.7086


### Avaliando o modelo

A avaliação é feita sobre o próprio dataset de treino, dado o tamanho reduzido do exemplo didático. O objetivo não é medir generalização mas sim verificar se o modelo aprendeu a distinção entre as duas classes e, principalmente, se a matriz S aprendida reflete a estrutura de comunidade esperada. 

Em grafos com comunidades, espera-se que os nós 0, 1 e 2 tenham alta probabilidade no mesmo cluster e os nós 3, 4 e 5 no outro. Em grafos homogêneos, as atribuições devem ser mais uniformes.

In [13]:
model.eval()
nomes = ["Com comunidades"]*3 + ["Homogêneo"]*3

with torch.no_grad():
    corretos = 0
    print(f"Atribuição de clusters S (cada linha = um nó)")
    for i, (A, X, y) in enumerate(dataset):
        out, S = model(A, X)
        pred = out.argmax().item()
        certo = "(correto)" if pred == y.item() else "(errado)"
        corretos += (pred == y.item())

        print(f"\nGrafo {i+1} ({nomes[i]}) | "
              f"Rótulo: {y.item()} | Predição: {pred} {certo}")
        print(S.round(decimals=2).numpy())

    print(f"\nAcurácia: {corretos}/{len(dataset)}")

Atribuição de clusters S (cada linha = um nó)

Grafo 1 (Com comunidades) | Rótulo: 1 | Predição: 1 (correto)
[[0.79 0.21]
 [0.79 0.21]
 [0.75 0.25]
 [0.63 0.37]
 [0.5  0.5 ]
 [0.48 0.52]]

Grafo 2 (Com comunidades) | Rótulo: 1 | Predição: 1 (correto)
[[0.79 0.21]
 [0.79 0.21]
 [0.75 0.25]
 [0.63 0.37]
 [0.5  0.5 ]
 [0.48 0.52]]

Grafo 3 (Com comunidades) | Rótulo: 1 | Predição: 1 (correto)
[[0.79 0.21]
 [0.79 0.21]
 [0.75 0.25]
 [0.63 0.37]
 [0.5  0.5 ]
 [0.48 0.52]]

Grafo 4 (Homogêneo) | Rótulo: 0 | Predição: 0 (correto)
[[0.7 0.3]
 [0.7 0.3]
 [0.7 0.3]
 [0.7 0.3]
 [0.7 0.3]
 [0.7 0.3]]

Grafo 5 (Homogêneo) | Rótulo: 0 | Predição: 0 (correto)
[[0.7  0.3 ]
 [0.7  0.3 ]
 [0.7  0.3 ]
 [0.71 0.29]
 [0.71 0.29]
 [0.71 0.29]]

Grafo 6 (Homogêneo) | Rótulo: 0 | Predição: 0 (correto)
[[0.69 0.31]
 [0.69 0.31]
 [0.69 0.31]
 [0.69 0.31]
 [0.69 0.31]
 [0.69 0.31]]

Acurácia: 6/6


O modelo acertou todos os 6 grafos, atingindo acurácia de 100% no dataset de treino. Mais interessante do que a acurácia, entretanto, é o que as matrizes $S$ revelam sobre o que o modelo aprendeu.

Nos grafos com comunidades, a matriz $S$ é idêntica entre os três, dado que os três grafos têm exatamente a mesma estrutura e _features_. 

Os nós 0, 1 e 2 (comunidade A) recebem atribuições de `[0.79, 0.21]` (aprox.), indicando forte preferência pelo cluster 0. Os nós 4 e 5 (comunidade B) têm atribuições próximas de `[0.50, 0.50]`, e o nó 3 fica em `[0.63, 0.37]`. 

O modelo separou as comunidades, mas de forma assimétrica; a comunidade A foi identificada com mais confiança do que a comunidade B. Os nós 3, 4 e 5, que deveriam ser claramente atribuídos ao cluster 1, mostram atribuições ainda ambíguas.

Nos grafos homogêneos (grafos 4, 5 e 6), todas as linhas de $S$ são virtualmente idênticas dentro de cada grafo, cerca de `[0.70/0.30]` em todos os nós. Isso confirma que o modelo não encontrou estrutura para diferenciar os nós entre si, todos recebem a mesma atribuição, independente de sua posição no grafo. Note que as atribuições não são exatamente `[0.50, 0.50]` como seria o caso de máxima incerteza. O modelo aprendeu um viés leve para o cluster 0, mas sem qualquer variação entre nós, o que é consistente com a ausência de comunidades.

# Nota: leia antes de continuar

A partir deste ponto do _notebook_, compreende-se que o leitor já está familiarizado com redes neurais de grafos e com o código do `pytorch`. Assim, não se dará tanto detalhamento na explicação dos códigos e dos resultados, assume-se que o leitor atento deverá os compreender de maneira aceitável.

# GCN

A Graph Convolutional Network (GCN), proposta por Kipf & Welling [7] é uma instância do _framework_ MPNN onde a função de mensagem $\psi$ é uma transformação linear e a agregação $\bigoplus$ é a média normalizada pelo grau. 

Originalmente motivada como uma aproximação de primeira ordem de convoluções espectrais [7], a GCN pode ser reinterpretada espacialmente como uma operação de _message passing_ local, tornando-a um ponto de convergência natural entre as abordagens espectrais e espaciais discutidas anteriormente.

A GCN é projetada para tarefas de nível de nó em cenário transdutivo, assim o modelo vê o grafo inteiro durante o treino, incluindo os nós de teste sem seus rótulos, e aprende a classificar cada nó com base em sua vizinhança.

## Como funciona

A regra de propagação de cada camada da GCN é [7]:

$$H^{(k)} = \sigma\left(\tilde{D}^{-\frac{1}{2}}\tilde{A}\tilde{D}^{-\frac{1}{2}}H^{(k-1)}W^{(k)}\right)$$

onde $\tilde{A} = A + I$ é a matriz de adjacência com auto-conexões, $\tilde{D}$ é a matriz de grau correspondente, $W^{(k)}$ é uma matriz de pesos aprendível e $\sigma$ é uma função de ativação. 

O termo $\tilde{D}^{-\frac{1}{2}}\tilde{A}\tilde{D}^{-\frac{1}{2}}$ normaliza a agregação pelo grau dos nós, evitando que nós com muitos vizinhos dominem a representação. 

Em termos do _framework_ MPNN, a GCN é uma instância convolucional onde os pesos $c_{u,v} = \frac{1}{\sqrt{d_u d_v}}$ são fixos e determinados pela estrutura do grafo [10].

## Função de Perda

Para classificação de nós, a GCN minimiza a entropia cruzada sobre os nós rotulados [7]:

$$\mathcal{L} = -\sum_{l \in \mathcal{Y}_L} \sum_{f=1}^{F} Y_{lf} \ln Z_{lf}$$

onde $\mathcal{Y}_L$ é o conjunto de nós com rótulos conhecidos e $Z$ é a saída _softmax_ do modelo. Note que o modelo é treinado apenas sobre os nós rotulados, mas a propagação de mensagens usa o grafo inteiro, incluindo os nós não rotulados, o que caracteriza o aprendizado semi-supervisionado transdutivo [7].

## Hiperparâmetros

| Hiperparâmetro | Descrição |
|---|---|
| $K$ | Número de camadas convolucionais |
| $d$ | Dimensão dos _embeddings_ ocultos |
| dropout | Taxa de _dropout_ entre camadas |
| $\lambda$ | Fator de regularização $\ell_2$ |

## Implementando com `pytorch`

### Criando os Dados

Trabalharemos com um grafo sintético de 10 nós dividido em duas comunidades, a mesma estrutura do DiffPool, agora usada para classificação de nós. O objetivo é atribuir a cada nó o rótulo de sua comunidade (0 ou 1), usando apenas alguns nós rotulados para treino.

In [14]:
A = torch.tensor([
    [0,1,1,1,0,0,0,0,0,0],
    [1,0,1,0,1,0,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,0],
    [1,0,1,0,1,0,0,0,0,0],
    [0,1,0,1,0,1,0,0,0,0],  # nó 4: ponte para comunidade 1
    [0,0,0,0,1,0,1,1,0,0],  # nó 5: ponte para comunidade 0
    [0,0,0,0,0,1,0,1,1,0],
    [0,0,0,0,0,1,1,0,0,1],
    [0,0,0,0,0,0,1,0,0,1],
    [0,0,0,0,0,0,0,1,1,0],
], dtype=torch.float)

X = torch.zeros(10, 2)
X[:5] = torch.tensor([1.0, 0.0]) + torch.randn(5, 2) * 0.1
X[5:] = torch.tensor([0.0, 1.0]) + torch.randn(5, 2) * 0.1

y = torch.tensor([0,0,0,0,0,1,1,1,1,1])

### Definindo as Camadas

A GCN é implementada como uma sequência de camadas convolucionais seguidas de uma camada de classificação. Cada camada convolucional implementa a regra de propagação de Kipf & Welling [7] diretamente, normalizando a adjacência pelo grau, multiplicando pelas features e aplicando ReLU. 

A última camada usa _softmax_ para produzir distribuições de probabilidade sobre as classes.

In [15]:
class CamadaGCN(nn.Module):
    """
    Camada convolucional da GCN [7].
    Instância de MPNN com agregação por média normalizada pelo grau
    e pesos fixos determinados pela estrutura do grafo.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, A_norm, X):
        # A_norm já é D^{-1/2} A~ D^{-1/2}, pré-computado
        return F.relu(self.W(A_norm @ X))


class GCN(nn.Module):
    """
    GCN para classificação de nós [7].

    Tarefa: classificar cada nó do grafo em uma das n_classes classes,
    usando apenas os nós rotulados para supervisão.
    """
    def __init__(self, in_dim, hidden_dim, n_classes, dropout=0.5):
        super().__init__()
        self.conv1 = CamadaGCN(in_dim, hidden_dim)
        self.conv2 = nn.Linear(hidden_dim, n_classes, bias=False)
        self.dropout = dropout

    def forward(self, A_norm, X):
        H = self.conv1(A_norm, X)
        H = F.dropout(H, p=self.dropout, training=self.training)
        out = self.conv2(A_norm @ H)
        return F.log_softmax(out, dim=-1)

    @staticmethod
    def normalizar_adjacencia(A):
        """
        Pré-computa D^{-1/2} (A + I) D^{-1/2}.
        """
        n = A.shape[0]
        A_t = A + torch.eye(n)
        D = A_t.sum(dim=1)
        D_inv_sqrt = torch.diag(D.pow(-0.5))
        return D_inv_sqrt @ A_t @ D_inv_sqrt

### Treinando o Modelo

O treino segue o protocolo semi-supervisionado de Kipf & Welling [7]. O _forward pass_ usa o grafo inteiro (incluindo nós não rotulados), mas a perda é computada apenas sobre os nós com rótulos conhecidos. Isso permite que a informação estrutural dos nós não rotulados influencie os _embeddings_ via _message passing_, mesmo sem supervisão direta.

In [16]:
mascara_treino = torch.tensor([True,True,False,False,False,
                               True,True,False,False,False])
mascara_teste = ~mascara_treino

A_norm = GCN.normalizar_adjacencia(A)

model = GCN(in_dim=2, hidden_dim=16, n_classes=2, dropout=0.3)
optimizer = Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

for epoch in range(1, 21):
    model.train()
    optimizer.zero_grad()
    out = model(A_norm, X)
    loss = F.nll_loss(out[mascara_treino], y[mascara_treino])
    loss.backward()
    optimizer.step()

    if epoch % 2 == 0:
        model.eval()
        with torch.no_grad():
            out_eval = model(A_norm, X)
            pred = out_eval.argmax(dim=1)
            acc_treino = (pred[mascara_treino] == y[mascara_treino]).float().mean()
            acc_teste = (pred[mascara_teste]  == y[mascara_teste]).float().mean()
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | "
              f"Acc treino: {acc_treino:.2f} | Acc teste: {acc_teste:.2f}")

Epoch   2 | Loss: 0.6796 | Acc treino: 0.50 | Acc teste: 0.50
Epoch   4 | Loss: 0.6210 | Acc treino: 0.50 | Acc teste: 0.50
Epoch   6 | Loss: 0.6071 | Acc treino: 1.00 | Acc teste: 0.83
Epoch   8 | Loss: 0.5874 | Acc treino: 1.00 | Acc teste: 0.83
Epoch  10 | Loss: 0.5702 | Acc treino: 1.00 | Acc teste: 1.00
Epoch  12 | Loss: 0.5590 | Acc treino: 1.00 | Acc teste: 1.00
Epoch  14 | Loss: 0.5224 | Acc treino: 1.00 | Acc teste: 1.00
Epoch  16 | Loss: 0.5284 | Acc treino: 1.00 | Acc teste: 1.00
Epoch  18 | Loss: 0.3935 | Acc treino: 1.00 | Acc teste: 1.00
Epoch  20 | Loss: 0.3899 | Acc treino: 1.00 | Acc teste: 1.00


### Avaliando o Modelo

In [17]:
model.eval()
with torch.no_grad():
    out = model(A_norm, X)
    pred = out.argmax(dim=1)

print("=== Avaliação por nó ===\n")
print(f"{'Nó':<5} {'Comunidade real':<18} {'Predição':<12} {'Treino/Teste':<14} {'Resultado'}")
print("-" * 65)
for i in range(10):
    real = y[i].item()
    p = pred[i].item()
    split = "Treino" if mascara_treino[i] else "Teste"
    certo = "(correto)" if p == real else "(errado)"
    print(f"{i:<5} {real:<18} {p:<12} {split:<14} {certo}")

acc_total = (pred == y).float().mean()
acc_teste = (pred[mascara_teste] == y[mascara_teste]).float().mean()
print(f"\nAcurácia total: {acc_total:.2f} | Acurácia nos nós de teste: {acc_teste:.2f}")

=== Avaliação por nó ===

Nó    Comunidade real    Predição     Treino/Teste   Resultado
-----------------------------------------------------------------
0     0                  0            Treino         (correto)
1     0                  0            Treino         (correto)
2     0                  0            Teste          (correto)
3     0                  0            Teste          (correto)
4     0                  0            Teste          (correto)
5     1                  1            Treino         (correto)
6     1                  1            Treino         (correto)
7     1                  1            Teste          (correto)
8     1                  1            Teste          (correto)
9     1                  1            Teste          (correto)

Acurácia total: 1.00 | Acurácia nos nós de teste: 1.00


# GraphSage + Link Prediction 

O GraphSAGE (SAmple and aggreGatE), proposto por Hamilton et al. [14], é uma arquitetura de GNN projetada para o cenário indutivo. Ao contrário da GCN, o GraphSAGE aprende uma função de agregação que generaliza para nós e grafos não vistos durante o treino, sem necessidade de reprocessar o grafo inteiro [14]. 

É uma instância do _framework_ MPNN onde a função de mensagem $\psi$ concatena a representação do nó com a agregação dos vizinhos, e $\bigoplus$ pode ser média, máximo ou LSTM [14]. Aqui, usamos o GraphSAGE para _link prediction_. 

## Como funciona?

O GraphSAGE gera _embeddings_ para cada nó em $K$ iterações [14]:

$$h_{\mathcal{N}(v)}^{(k)} = \text{AGGREGATE}_k\left(\{h_u^{(k-1)}, \forall u \in \mathcal{N}(v)\}\right)$$

$$h_v^{(k)} = \sigma\left(W^{(k)} \cdot \text{CONCAT}\left(h_v^{(k-1)}, h_{\mathcal{N}(v)}^{(k)}\right)\right)$$

A concatenação do _embedding_ atual do nó com a agregação dos vizinhos é a diferença central em relação à GCN, ela preserva a identidade do nó ao longo das camadas, evitando o _oversmoothing_ que ocorre em GCNs profundas [14]. Nesta implementação, usamos a variante com agregação por média, a mais simples e a que mais se aproxima da GCN [14].

Para _link prediction_, os _embeddings_ dos nós são usados para pontuar pares de nós. A pontuação de uma aresta $(u, v)$ é o produto interno entre os _embeddings_:

$$\hat{y}_{uv} = \sigma\left(h_u^{(K)^T} h_v^{(K)}\right)$$

## Função de Perda

A perda de _link prediction_ é a _binary cross-entropy_ entre as arestas existentes (positivas) e arestas inexistentes amostradas aleatoriamente (negativas) [14]:

$$\mathcal{L} = -\sum_{(u,v) \in E} \log \hat{y}_{uv} - \sum_{(u,v) \notin E} \log(1 - \hat{y}_{uv})$$

## Hiperparâmetros

| Hiperparâmetro | Descrição |
|---|---|
| $K$ | Número de camadas de agregação |
| $d$ | Dimensão dos _embeddings_ |
| neg_ratio | Número de arestas negativas por aresta positiva |
| dropout | Taxa de _dropout_ entre camadas |

## Implementando com `pytorch`

### Criando os Dados

Trabalharemos com um grafo sintético de 16 nós dividido em duas comunidades, dois cliques de 8 nós cada, conectados por uma única aresta ponte entre os nós 7 e 8. Um clique é um subgrafo onde todos os nós são vizinhos entre si, o que maximiza a densidade interna de cada comunidade e torna a distinção entre arestas intra e intercomunidade mais clara. 

O grafo é maior do que o usado nas seções anteriores pois _link prediction_ requer um número razoável de arestas positivas e negativas para treino e teste.

As features têm dimensão 8; os nós da comunidade 0 têm valores altos nas primeiras 4 dimensões e baixos nas últimas 4, e os nós da comunidade 1 têm o padrão inverso. Essa separação no espaço de _features_ complementa a separação topológica, dando ao modelo dois sinais independentes para aprender a distinção entre pares de nós da mesma comunidade e pares de comunidades diferentes.

As arestas negativas são geradas explicitamente como todos os pares $(i, j)$ com $i < j$ onde $A_{ij} = 0$, garantindo que não haja sobreposição com as arestas positivas. Um subconjunto aleatório de tamanho igual ao número de arestas positivas é amostrado para manter o _dataset_ balanceado.

In [18]:
n = 16
A = torch.zeros(n, n)

def fazer_clique(nos):
    for i in nos:
        for j in nos:
            if i != j:
                A[i, j] = 1.0

fazer_clique(range(8)) 
fazer_clique(range(8, 16)) 
A[7, 8] = 1.0 
A[8, 7] = 1.0 

torch.manual_seed(33)
X = torch.zeros(n, 8)
X[:8,  :4] = 1.0 + torch.randn(8, 4) * 0.1
X[8:, 4:] = 1.0 + torch.randn(8, 4) * 0.1

arestas_pos = torch.stack(
    torch.where(torch.triu(A, diagonal=1) == 1), dim=1
)

arestas_neg_candidatas = []
for i in range(n):
    for j in range(i+1, n):
        if A[i, j] == 0:
            arestas_neg_candidatas.append([i, j])

arestas_neg_todas = torch.tensor(arestas_neg_candidatas)
torch.manual_seed(33)
idx = torch.randperm(len(arestas_neg_todas))[:len(arestas_pos)]
arestas_neg = arestas_neg_todas[idx]

n_pos = len(arestas_pos)
n_treino = int(n_pos * 0.7)

arestas_pos_treino = arestas_pos[:n_treino]
arestas_pos_teste  = arestas_pos[n_treino:]
arestas_neg_treino = arestas_neg[:n_treino]
arestas_neg_teste  = arestas_neg[n_treino:]

print(f"Nós: {n} | Arestas positivas: {n_pos}")
print(f"Treino: {n_treino} pos + {n_treino} neg | "
      f"Teste: {n_pos-n_treino} pos + {n_pos-n_treino} neg")

Nós: 16 | Arestas positivas: 57
Treino: 39 pos + 39 neg | Teste: 18 pos + 18 neg


### Definindo as Camadas

O modelo é composto por duas classes. A classe base é `CamadaGraphSAGE`, que implementa a regra de propagação do GraphSAGE com agregação por média [14]. A diferença central em relação à GCN está na linha `torch.cat([X, agg], dim=-1)`; em vez de substituir a representação do nó pela agregação dos vizinhos, o GraphSAGE concatena as duas — preservando a identidade do nó ao longo das camadas e evitando o _oversmoothing_ que ocorre em GCNs profundas [14]. Por isso, a camada linear recebe `in_dim * 2` como dimensão de entrada.

`GraphSAGE` empilha duas camadas e adiciona um _decoder_ MLP para pontuar pares de nós. O _decoder_ recebe a concatenação dos _embeddings_ dos dois nós candidatos a uma aresta e aprende diretamente a função de _scoring_, isso é mais expressivo do que o produto interno simples, pois permite que o modelo aprenda relações assimétricas e não-lineares entre os _embeddings_. A sigmoide na saída de `pontuar_arestas` converte o _score_ em uma probabilidade entre 0 e 1.

In [19]:
class CamadaGraphSAGE(nn.Module):
    """
    Camada GraphSAGE com agregação por média [14].
    Instância de MPNN onde a mensagem concatena o nó com
    a média de seus vizinhos antes da transformação linear.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # in_dim * 2 pois concatenamos h_v com h_N(v)
        self.W = nn.Linear(in_dim * 2, out_dim, bias=True)

    def forward(self, A, X):
        deg = A.sum(dim=-1, keepdim=True).clamp(min=1)
        agg = (A @ X) / deg
        H = torch.cat([X, agg], dim=-1)
        return F.relu(self.W(H))


class GraphSAGE(nn.Module):
    """
    GraphSAGE para link prediction [14].

    Tarefa: dado um grafo, prever se uma aresta existe
    entre dois nós com base em seus embeddings.
    """
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.2):
        super().__init__()
        self.conv1 = CamadaGraphSAGE(in_dim, hidden_dim)
        self.conv2 = CamadaGraphSAGE(hidden_dim, out_dim)
        self.dropout = dropout
        # Decoder MLP: concatena embeddings dos dois nós e pontua o par
        self.decoder = nn.Sequential(
            nn.Linear(out_dim * 2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, A, X):
        H = self.conv1(A, X)
        H = F.dropout(H, p=self.dropout, training=self.training)
        H = self.conv2(A, H)
        return H

    def pontuar_arestas(self, H, arestas):
        """
        Pontua pares de nós via decoder MLP — concatena os embeddings
        dos dois nós e aprende a função de scoring diretamente.
        """
        u, v = arestas[:, 0], arestas[:, 1]
        par = torch.cat([H[u], H[v]], dim=-1)
        score = self.decoder(par).squeeze(-1)
        return torch.sigmoid(score)

### Treinando o Modelo

O treino segue o protocolo de _link prediction_ de Hamilton et al. [14]. A cada iteração, o modelo recebe pares de arestas positivas e negativas e minimiza a Entropia Cruzada Binária entre os _scores_ preditos e os rótulos reais (1 para arestas existentes e 0 para pares sem aresta). 

Diferente da GCN, não há distinção entre nós rotulados e não rotulados, toda a supervisão vem da estrutura de arestas. O modelo processa o grafo inteiro a cada passo e gera _embeddings_ para todos os nós, que são então usados pelo _decoder_ para pontuar os pares de interesse.

In [20]:
model = GraphSAGE(in_dim=8, hidden_dim=32, out_dim=16)
optimizer = Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()

    H = model(A, X)
    scores_pos = model.pontuar_arestas(H, arestas_pos_treino)
    scores_neg = model.pontuar_arestas(H, arestas_neg_treino)

    labels = torch.cat([torch.ones(len(arestas_pos_treino)),
                        torch.zeros(len(arestas_neg_treino))])
    scores = torch.cat([scores_pos, scores_neg])
    loss = F.binary_cross_entropy(scores, labels)
    loss.backward()
    optimizer.step()

    if epoch % 40 == 0:
        model.eval()
        with torch.no_grad():
            H_eval = model(A, X)
            s_pos = model.pontuar_arestas(H_eval, arestas_pos_teste)
            s_neg = model.pontuar_arestas(H_eval, arestas_neg_teste)
            pred_pos = (s_pos > 0.5).float()
            pred_neg = (s_neg < 0.5).float()
            acc = torch.cat([pred_pos, pred_neg]).mean()
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Acc teste: {acc:.2f}")

Epoch  40 | Loss: 0.0373 | Acc teste: 1.00
Epoch  80 | Loss: 0.0397 | Acc teste: 1.00
Epoch 120 | Loss: 0.0110 | Acc teste: 1.00
Epoch 160 | Loss: 0.0289 | Acc teste: 1.00
Epoch 200 | Loss: 0.0274 | Acc teste: 1.00


### Avaliando o Modelo

A avaliação exibe o _score_ de cada par de nós no conjunto de teste, separando arestas positivas (_score_ acima de 0.5) das negativas (_score_ abaixo de 0.5). 

Um modelo que aprendeu a estrutura do grafo deve atribuir _scores_ próximos de 1.0 a pares de nós dentro da mesma comunidade, onde as arestas de fato existem, e _scores_ próximos de 0.0 a pares entre comunidades diferentes, onde não há conexão. 

A ponte entre os nós 7 e 8 é um caso especial, conecta nós de comunidades distintas, e o modelo precisa aprender a tratá-la como positiva apesar do perfil de _features_ divergente.

In [21]:
model.eval()
with torch.no_grad():
    H = model(A, X)
    s_pos = model.pontuar_arestas(H, arestas_pos_teste)
    s_neg = model.pontuar_arestas(H, arestas_neg_teste)

print("Arestas positivas\n")
for aresta, score in zip(arestas_pos_teste, s_pos):
    u, v = aresta.tolist()
    certo = "(correto)" if score > 0.5 else "(errado)"
    print(f"Aresta ({u:2d},{v:2d}) | Score: {score:.3f} {certo}")

print("\nArestas negativas\n")
for aresta, score in zip(arestas_neg_teste, s_neg):
    u, v = aresta.tolist()
    certo = "(correto)" if score < 0.5 else "(correto)"
    print(f"Par    ({u:2d},{v:2d}) | Score: {score:.3f} {certo}")

pred_pos = (s_pos > 0.5).float()
pred_neg = (s_neg < 0.5).float()
acc = torch.cat([pred_pos, pred_neg]).mean()
print(f"\nAcurácia no conjunto de teste: {acc:.2f}")

Arestas positivas

Aresta ( 9,13) | Score: 1.000 (correto)
Aresta ( 9,14) | Score: 1.000 (correto)
Aresta ( 9,15) | Score: 1.000 (correto)
Aresta (10,11) | Score: 1.000 (correto)
Aresta (10,12) | Score: 1.000 (correto)
Aresta (10,13) | Score: 1.000 (correto)
Aresta (10,14) | Score: 1.000 (correto)
Aresta (10,15) | Score: 1.000 (correto)
Aresta (11,12) | Score: 1.000 (correto)
Aresta (11,13) | Score: 1.000 (correto)
Aresta (11,14) | Score: 1.000 (correto)
Aresta (11,15) | Score: 1.000 (correto)
Aresta (12,13) | Score: 1.000 (correto)
Aresta (12,14) | Score: 1.000 (correto)
Aresta (12,15) | Score: 1.000 (correto)
Aresta (13,14) | Score: 1.000 (correto)
Aresta (13,15) | Score: 1.000 (correto)
Aresta (14,15) | Score: 1.000 (correto)

Arestas negativas

Par    ( 6,14) | Score: 0.000 (correto)
Par    ( 4,13) | Score: 0.000 (correto)
Par    ( 1, 8) | Score: 0.005 (correto)
Par    ( 0,14) | Score: 0.000 (correto)
Par    ( 5,13) | Score: 0.000 (correto)
Par    ( 7, 9) | Score: 0.007 (correto)
P

# Referências

1. Notebook "LMA-103 6.0 - Grafos" do professor Dr. Daniel R. Cassar da disciplina "Lógica Computacional", ministrada em 2025.
2. Grafos (Matemática Discreta) https://en.wikipedia.org/wiki/Graph_(discrete_mathematics).
3. A Gentle Introduction to Graph Neural Networks (artigo web) de Sanchez-Lengeling, et al. $-$ disponível em https://distill.pub/2021/gnn-intro/
4. Mastering Graph Neural Networks: Exploring GNN Types, Tasks, and Top Models (artigo web) de AJ $-$ disponível em https://medium.com/@adityaj5400/mastering-graph-neural-networks-exploring-gnn-types-tasks-and-top-models-bbe09e37447d
5. The link-prediction problem for social networks (artigo) de Liben-Nowell, D. and Kleinberg, J (2007) $-$ disponível em https://doi.org/10.1002/asi.20591
6. Capítulo 13 $-$ "Graph neural networks" $-$ do livro Understanding Deep Learning de Simon J. D. Prince, publicado em 2023 pela "The MIT Press"; disponível em http://udlbook.com
7. Semi-Supervised Classification With Graph Convolutional Networks (artigo) de Kipf, T. N.  e Welling, M. $-$ disponível em https://arxiv.org/abs/1609.02907
8. Transformers are Graph Neural Networks (artigo web) de Joshi, C. (2020) $-$ disponível em https://thegradient.pub/transformers-are-graph-neural-networks/
9. A Generalization of Transformers to Graphs (artigo) de Dwivedi, V. P. e Bresson, X. (2020) $-$ disponível em https://arxiv.org/abs/2012.09699
10. Graph Neural Networks Designed for Different Graph Types: A Survey (artigo) de Josephine, et al (2023) $-$ disponível em https://arxiv.org/pdf/2204.03080
11. Hierarchical Graph Representation Learning with Differentiable Pooling (artigo) de Ying, R. et al. (2018) $-$ disponível em https://arxiv.org/abs/1806.08804
12. How Powerful are Graph Neural Networks? (artigo) de Xu et al. (2019) $-$ disponível em https://arxiv.org/abs/1810.00826
13. Neural Message Passing for Quantum Chemistry (artigo) de Gilmer et al. (2017) $-$ disponivel em https://arxiv.org/pdf/1704.01212
14. Inductive Representation Learning on Large Graphs de Hamilton et al. (2017) $-$ disponível em https://arxiv.org/abs/1706.02216